In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('cleaned_data.csv')

In [4]:
df = df.drop('kms_group', axis=1)

In [6]:
#Lets divide dataset into X and y
X = df[['company', 'name', 'year', 'kms_driven', 'fuel_type']]
y = df[['Price']]

In [16]:
#Lets create pipeline which consists of columntransformer - whose job is to give numerical columns
#for string columns using OneHotEncoder and it returns other numerical columns as it is.
#After column transformer we create model object and put it in pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

ohe = OneHotEncoder()
ohe.fit(X[["company", "name", "fuel_type"]])
ct = make_column_transformer((OneHotEncoder(categories = ohe.categories_), 
                              ["company", "name", "fuel_type"]), remainder = 'passthrough')
model = LinearRegression()

pipe = make_pipeline(ct, model)

In [34]:
#Lets divide dataset into training and testing

from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

scores = []

for i in range(0, 101):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.05, random_state = i)
    #Lets train pipeline
    pipe.fit(X_train, y_train)
    #Lets find r2_score
    y_pred = pipe.predict(X_test)
    y_pred = pd.DataFrame(data = y_pred, columns = ['Prediction'])
    result = pd.concat([y_test.reset_index(drop = True), y_pred], axis = 1)
    score = r2_score(result['Price'], result['Prediction'])
    scores.append(score)

In [36]:
#Find index of max score
index = np.argmax(scores)

np.int64(19)

In [38]:
#Lets retrain with best index
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.05, random_state = index)
pipe.fit(X_train, y_train)

,steps,"[('columntransformer', ...), ('linearregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehotencoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [45]:
data = [['Ford', 'EcoSport', 2015, 80000, 'Petrol']]
columns = ['company', 'name', 'year', 'kms_driven', 'fuel_type']
myinput = pd.DataFrame(data = data, columns = columns)
price = pipe.predict(myinput)
print("You can purchase this car for:", round(price[0,0]))

You can purchase this car for: 547703


In [47]:
#To get input from user and provide output we must develop web application
#We can use Python - Flask, FastAPI, Django or Streamlit
#We cannot train model all the time, so lets export out trained model(pipe) using pickle library
import pickle as pkl
pkl.dump(pipe, open('car-price-predictor.pkl', 'wb'))